 ### Importing

In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms,datasets
if torch.cuda.is_available():
    device=torch.device("cpu")
    device = torch.device("cuda") 
print(f"Using device: {device}")

Using device: cuda


 ### Data Pre-Processing

In [6]:
transform=transforms.ToTensor()
train_dataset=datasets.MNIST(train=True,transform=transform,root="data",download=True)
test_dataset=datasets.MNIST(train=False,transform=transform,root="data",download=True)
train_loader=DataLoader(torch.utils.data.Subset(train_dataset, range(10000)),shuffle=True,batch_size=64)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=True)
x,y=(next(iter(train_loader)))
print(x.shape)
print(y.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


 ### Model

In [19]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=1,padding=1)
        self.batchn1=nn.BatchNorm2d(16)
        self.pool=nn.MaxPool2d(kernel_size=2)
        self.conv2=nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,stride=1,padding=1)
        self.batchn2=nn.BatchNorm2d(32)
        self.fc=nn.Linear(7*7*32,10)#28*28*32,10)
    def forward(self,x):
        x=self.conv1(x)
        #print(x.shape)

        x=self.batchn1(x)
        #print(x.shape)

        x=torch.relu(x)
        #print(x.shape)
        
        x=self.pool(x)
        #print(x.shape)

        x=self.conv2(x)
        #print(x.shape)
        
        x=self.batchn2(x)     
        #print(x.shape)

        x=torch.relu(x)
        #print(x.shape)

        x=self.pool(x)
        #print(x.shape)

        x=torch.flatten(x,1)
        #print(x.shape)
        
        x=self.fc(x)
        #print(x.shape)
        return x

In [21]:
epoch=10
model=CNN()
model = model.to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
loss_fn=nn.CrossEntropyLoss()
model.train()
for i in range(epoch):
    loss=0
    for image,label in train_loader:
        image=image.to(device)
        label=label.to(device)
        model.zero_grad()
        predicted_result=model.forward(image)
        batch_loss=loss_fn(predicted_result,label)
        batch_loss.backward()
        optimizer.step()
        loss+=batch_loss.item()
    if i%2==0:
        print(loss/len(train_loader))

0.6604068645626117
0.07712702218081303
0.04600400301319351
0.02726351938122931
0.010655856036371996


### Loss sequence without pooling:
2.8104847313681987<br>
0.042484355203964555<br>
0.02142438677508105<br>
0.0275420495401623<br>
0.01046086661656651<br>